In [ ]:
from sqlalchemy import create_engine
from sqlalchemy import URL
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from geopy.geocoders import Nominatim

In [2]:
server = 'sharktowels.duckdns.org'
database = "Spending"
user = "SA"

with open("password.txt", "r") as file:
    password = file.read().strip()

conn_string = URL.create(
    "mssql+pyodbc",
    username = user,
    password = password,
    host = server,
    port = 1433,
    database = database,
    query = {
            "driver": "ODBC Driver 18 for SQL Server",
            "TrustServerCertificate": "yes",
            }
)

conn = create_engine(conn_string)


In [3]:
#ONLY FOR NOTEBOOK SIDE TOKEN FETCHING
df = pd.read_sql_query(f'''
                        SELECT PasswordHash FROM Credentials
                        ''', conn)

print(df)

temptoken = str(df.iloc[0,0])
print(temptoken)
token = temptoken

                                        PasswordHash
0  e7cf3ef4f17c3999a94f2c6f612e8a888e5b1026878e4e...
e7cf3ef4f17c3999a94f2c6f612e8a888e5b1026878e4e19398b23bd38ec221a


c:\Users\jduen\AppData\Local\Programs\Python\Python311\Lib\site-packages\pandas\io\sql.py:1636: SAWarning: Unrecognized server version info '17.0.1000.7'.  Some SQL Server features may not function properly.
  con = self.exit_stack.enter_context(con.connect())


In [4]:
df = pd.read_sql_query(f'''
                        SELECT Users.UserID
                        FROM Users
                        JOIN Credentials ON Users.UserID = Credentials.UserID
                        WHERE Credentials.PasswordHash = '{token}'
                        ''', conn)

userid = str(df.iloc[0,0])

In [7]:
#number 2
area_df = pd.read_sql_query(f'''
                            SELECT Purchases.TimeDate, Purchases.Category, Payments.Amount
                            FROM Purchases
                            JOIN Payments ON Purchases.PaymentID = Payments.PaymentID
                            JOIN Users ON Users.UserID = Purchases.UserID
                            WHERE Users.UserID = '{userid}'
''', conn)


area_df = area_df.sort_values("TimeDate").copy()
now = area_df["TimeDate"].max()

timeframes = {
    "1m":  pd.DateOffset(months=1),
    "3m":  pd.DateOffset(months=3),
    "6m":  pd.DateOffset(months=6),
    "1y":  pd.DateOffset(years=1),
    "ytd": "ytd",
    "all": "all"
}

all_traces = []
trace_counts = []

for label, offset in timeframes.items():

    if offset == "all":
        df = area_df.copy()
    elif offset == "ytd":
        df = area_df[area_df["TimeDate"].dt.year == now.year].copy()
    else:
        df = area_df[area_df["TimeDate"] >= now - offset].copy()

    df = df.sort_values("TimeDate")

    if df.empty:
        trace_counts.append(0)
        continue

    df["runningAmount"] = (
        df.groupby("Category")["Amount"]
          .cumsum()
    )

    pivoted = df.pivot_table(
        index="TimeDate",
        columns="Category",
        values="runningAmount",
        aggfunc="last"
    ).ffill()

    first = True
    count = 0
    for category in pivoted.columns:
        all_traces.append(
            go.Scatter(
                x=pivoted.index,
                y=pivoted[category],
                mode="lines",
                stackgroup="one",
                name=f"{category} ({label})",
                visible=False
            )
        )
        first = False
        count += 1

    trace_counts.append(count)


fig = go.Figure()

for tr in all_traces:
    fig.add_trace(tr)

buttons = []
start = 0

for (label, _), count in zip(timeframes.items(), trace_counts):

    mask = [False] * len(all_traces)
    for i in range(start, start + count):
        mask[i] = True

    buttons.append(
        dict(
            label=label.upper(),
            method="update",
            args=[
                {"visible": mask},
                {"title": f"Cumulative Spending: {label.upper()}"}
            ]
        )
    )

    start += count

fig.update_layout(
    updatemenus=[
        dict(
            buttons=buttons,
            direction="down",
            x=1.15,
            y=1.0,
            showactive=True
        )
    ],
    title="Cumulative Spending: 1M",
    xaxis_title="Time",
    yaxis_title="Running Amount",
    hovermode="x unified"
)

initial_count = trace_counts[0]
for i in range(initial_count):
    fig.data[i].visible = True

fig.show()


In [10]:
#number 4
pie_df = pd.read_sql_query(f'''
                            SELECT Purchases.Category, Purchases.Subcategory, Payments.Amount
                            FROM Purchases
                            JOIN Payments ON Purchases.PaymentID = Payments.PaymentID
                            WHERE UserID = {userid}
                            ORDER BY Purchases.TimeDate
                            ''', conn)

grouped_amount = pie_df.groupby(["Category", "Subcategory"])["Amount"].sum().reset_index(name="total")
grouped_count = pie_df.groupby(["Category", "Subcategory"]).size().reset_index(name="count")

fig = make_subplots(rows = 1, cols = 2,
                     specs=[[{"type": "domain"}, {"type": "domain"}]],
                     subplot_titles=("Total Amount Spent", "Number of Purchases"))

categories = pie_df["Category"].unique()

for i, category in enumerate(categories):
    category_amount_df = grouped_amount[grouped_amount["Category"] == category]
    category_count_df = grouped_count[grouped_count["Category"] == category]
    
    fig.add_trace(

        go.Pie(
            labels = category_amount_df["Subcategory"],
            values = category_amount_df["total"],
            name = f"{category}: Count",
            visible = (i == 0),
            hovertemplate = "%{label}: $%{value}<extra></extra>"
        ), 
        row = 1, col = 1
    )

    fig.add_trace(

        go.Pie(
            labels = category_count_df["Subcategory"],
            values = category_count_df["count"],
            name = f"{category}: Amount",
            visible = (i == 0),
            hovertemplate = "%{label}, %{value}<extra></extra>"
        ),
        row = 1, col = 2
    )

fig.update_layout(

    updatemenus=[
        dict(
            buttons = [
                dict(
                    label = category,
                    method = "update",
                    args = [
                        {"visible": [(j // 2) == i for j in range(2 * len(categories))]},
                        {"title": f"Purchase Breakdown: {category}"}
                    ]
                )
                for i, category in enumerate(categories)
            ],
            direction = "down",
            x = 1,
            y = 1,
        )

    ],
    title = f"Purchase Breakdown: {categories[0]}"
)

fig.show()
    

In [8]:
#number 3
category_bar_df = pd.read_sql_query(f'''
                  SELECT Purchases.Category, Purchases.Subcategory, Payments.Amount
                  FROM Purchases
                  JOIN Payments ON Purchases.PaymentID = Payments.PaymentID
                  WHERE Purchases.UserID = {userid}
                  ORDER BY Purchases.Category
                  ''', conn)

fig = px.bar(category_bar_df, x = "Category", y = "Amount", color = "Subcategory", title = "Spending per Category")
fig.show()

In [ ]:
#number 1
df = pd.read_sql_query(f'''

                        SELECT TOP 5 Purchases.TimeDate, Purchases.Location, Purchases.Category, Purchases.Subcategory, Payments.Amount, Payments.Type
                        FROM Purchases
                        JOIN Payments ON Purchases.PaymentID = Payments.PaymentID
                        JOIN Users ON Purchases.UserID = Users.UserID
                        WHERE Users.UserID = {userid}
                        ORDER BY Purchases.TimeDate DESC
                        ''', conn)

fig = go.Figure(data=[go.Table(
    header=dict(values=['Time/Date', 'Location', 'Category', 'SubCategory', 'Amount', 'Type'],
                align='center'),
    cells=dict(values=[df.TimeDate, df.Location, df.Category, df.Subcategory, df.Amount, df.Type],
               align='left')),
])

fig.update_layout(title_text='Your Last 5 Purchases')

fig.show()

In [17]:
#the last one
geolocator = Nominatim(user_agent="my_geocoder_app")
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)

location_df = pd.read_sql_query(f'''
                        SELECT Purchases.TimeDate, Purchases.Location, Purchases.Category, Payments.Amount
                        FROM Purchases
                        JOIN Payments ON Purchases.PaymentID = Payments.PaymentID
                        JOIN Users ON Purchases.UserID = Users.UserID
                        WHERE Users.UserID = {userid}
                        ORDER BY Purchases.TimeDate
                        ''', conn)

lonlat = []

def get_lonlat(location):
    index = location.find(',')
    city = location[:index]
    state = location[-2:]

    try:
        geolocation = geolocator.geocode(f"{city}, {state}")
        return (geolocation.latitude, geolocation.longitude)
    except:
        return (None, None)
    

location_df[["latitude", "longitude"]] = location_df["Location"].apply(lambda x: pd.Series(get_lonlat(x)))

fig = px.scatter_geo(location_df, lat = "latitude", lon = "longitude", color = "Category", size = "Amount", projection = 'albers usa')

fig.show()

